In [ ]:
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
import matplotlib
from matplotlib import pyplot as plt
import requests

In [ ]:
url = 'https://raw.githubusercontent.com/ZioloPan/AGH_Advanced-vision-algorithms/main/lab2/'
file_names = [
    'highway.tar.xz',
    'office.tar.xz',
    'pedestrian.tar.xz'
]

for file_name in file_names:
  r = requests.get(url + file_name, allow_redirects=True)
  open(file_name, 'wb').write(r.content)

for file_name in file_names:
    !tar -xf {file_name}

#3.1

In [ ]:
def post_process(diff):
    # 1. Binaryzacja
    _,result = cv2.threshold(diff.astype('uint8'), 10, 255, cv2.THRESH_BINARY)

    # 2. medianBlur
    # result = cv2.medianBlur(result, 5)

    # 3. zamknięcie
    kernel = np.ones((3,3), np.uint8)
    result = cv2.erode(result, kernel, iterations=1)
    result = cv2.dilate(result, kernel, iterations=1)

    return result

In [ ]:
def show(IG, model_tla, diff, mask):
    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.title("IG")
    plt.imshow(IG, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.title("Model tła")
    plt.imshow(model_tla, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.title("Różnica")
    plt.imshow(diff, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.title("Maska")
    plt.imshow(mask, cmap='gray')
    plt.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
def calculate_metrics(mask_pred, mask_gt):
    TP = np.sum(np.logical_and(mask_pred == 255, mask_gt == 255))
    FP = np.sum(np.logical_and(mask_pred == 255, mask_gt == 0))
    FN = np.sum(np.logical_and(mask_pred == 0,   mask_gt == 255))
    return TP, FP, FN

In [ ]:
def summarize_metrics(TP, FP, FN, name):
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    F1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"\n{name} METRICS:")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {F1:.4f}")

In [ ]:
roi_path = 'pedestrian/temporalROI.txt'
with open(roi_path, 'r') as f:
    line = f.readline()
    roi_start, roi_end = line.split()
    roi_start = int(roi_start)
    roi_end = int(roi_end)

start_frame = cv2.imread(f"pedestrian/input/in000001.jpg")
start_frame_gray = cv2.cvtColor(start_frame, cv2.COLOR_BGR2GRAY)
start_n = 400
XX = start_frame.shape[1]
YY = start_frame.shape[0]
N = 60

BUF = np.zeros((YY, XX, N), np.uint8)
iN = 0

TP = FP = FN = 0

for i in range(start_n, start_n+N, 1):
      current_frame_bgr = cv2.imread(f"pedestrian/input/in{i:06d}.jpg")
      IG = cv2.cvtColor(current_frame_bgr, cv2.COLOR_BGR2GRAY)

      BUF[:, :, iN] = IG

      iN += 1
      if iN >= N:
          iN = 0

      # -----------------------------------------
      # 1. Obliczanie tła metodą ŚREDNIEJ
      model_tla = np.mean(BUF, axis=2)
      model_tla = model_tla.astype(np.uint8)

      # 2. Obliczanie tła metodą MEDIANY
      # model_tla = np.median(BUF, axis=2)
      # model_tla = model_tla.astype(np.uint8)
      # -----------------------------------------

      diff = cv2.absdiff(IG, model_tla)

      mask = post_process(diff)

      # --- EWALUACJA ---
      if roi_start <= i <= roi_end:
          GT = cv2.imread(f"pedestrian/groundtruth/gt{i:06d}.png", cv2.IMREAD_GRAYSCALE)
          if GT is not None:
              tp, fp, fn = calculate_metrics(mask, GT)
              TP += tp
              FP += fp
              FN += fn

      show(IG, model_tla, diff, mask)

summarize_metrics(TP, FP, FN, "BUF")

#3.2

In [ ]:
def show_all_in_one_row(
    IG,
    BG_average,
    diff_average,
    mask_average,
    BG_median,
    diff_median,
    mask_median
):
    plt.figure(figsize=(20, 4))

    plt.subplot(1, 7, 1)
    plt.title("IG")
    plt.imshow(IG, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 7, 2)
    plt.title("BG avg")
    plt.imshow(BG_average.astype(np.uint8), cmap='gray')
    plt.axis('off')

    plt.subplot(1, 7, 3)
    plt.title("diff avg")
    plt.imshow(diff_average, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 7, 4)
    plt.title("mask avg")
    plt.imshow(mask_average, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 7, 5)
    plt.title("BG median")
    plt.imshow(BG_median, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 7, 6)
    plt.title("diff median")
    plt.imshow(diff_median, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 7, 7)
    plt.title("mask median")
    plt.imshow(mask_median, cmap='gray')
    plt.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
roi_path = 'pedestrian/temporalROI.txt'
with open(roi_path, 'r') as f:
    line = f.readline()
    roi_start, roi_end = line.split()
    roi_start = int(roi_start)
    roi_end = int(roi_end)

alpha = 0.01
start_idx = 300
end_idx = 600

first_frame_bgr = cv2.imread(f"pedestrian/input/in{start_idx:06d}.jpg")
first_frame_gray = cv2.cvtColor(first_frame_bgr, cv2.COLOR_BGR2GRAY)


BG_average = first_frame_gray.astype(np.float64)
BG_median = first_frame_gray.copy()
TP_avg = FP_avg = FN_avg = 0
TP_med = FP_med = FN_med = 0


for i in range(start_idx + 1, end_idx + 1):
    frame_bgr = cv2.imread(f"pedestrian/input/in{i:06d}.jpg")

    IG = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)


    # Metoda ŚREDNIEJ
    BG_average = alpha * IG + (1 - alpha) * BG_average
    diff_average = cv2.absdiff(IG, BG_average.astype(np.uint8))
    _,mask_average = cv2.threshold(diff_average, 10, 255, cv2.THRESH_BINARY)

    # Metoda 'MEDIANY'
    BG_median = BG_median + (BG_median < IG).astype(np.uint8) \
                            - (BG_median > IG).astype(np.uint8)
    diff_median = cv2.absdiff(IG, BG_median)
    _, mask_median = cv2.threshold(diff_median, 10, 255, cv2.THRESH_BINARY)

    # --- EWALUACJA ---
    if roi_start <= i <= roi_end:
        GT = cv2.imread(f"pedestrian/groundtruth/gt{i:06d}.png", cv2.IMREAD_GRAYSCALE)
        if GT is not None:
            tp, fp, fn = calculate_metrics(mask_average, GT)
            TP_avg += tp
            FP_avg += fp
            FN_avg += fn

            tp, fp, fn = calculate_metrics(mask_median, GT)
            TP_med += tp
            FP_med += fp
            FN_med += fn

    show_all_in_one_row(
        IG,
        BG_average,
        diff_average,
        mask_average,
        BG_median,
        diff_median,
        mask_median
    )

summarize_metrics(TP_avg, FP_avg, FN_avg, "AVERAGE")
summarize_metrics(TP_med, FP_med, FN_med, "MEDIAN")

#3.3

In [ ]:
roi_path = 'pedestrian/temporalROI.txt'
with open(roi_path, 'r') as f:
    line = f.readline()
    roi_start, roi_end = line.split()
    roi_start = int(roi_start)
    roi_end = int(roi_end)

alpha = 0.01
start_idx = 300
end_idx = 600

first_frame_bgr = cv2.imread(f"pedestrian/input/in{start_idx:06d}.jpg")
first_frame_gray = cv2.cvtColor(first_frame_bgr, cv2.COLOR_BGR2GRAY)

BG_average_cons = first_frame_gray.astype(np.float64)
TP_avg_cons = FP_avg_cons = FN_avg_cons = 0

BG_median_cons = first_frame_gray.copy()
TP_med_cons = FP_med_cons = FN_med_cons = 0

mask_prev_avg_cons = np.zeros_like(first_frame_gray, dtype=np.uint8)
mask_prev_med_cons = np.zeros_like(first_frame_gray, dtype=np.uint8)

for i in range(start_idx + 1, end_idx + 1):
    frame_bgr = cv2.imread(f"pedestrian/input/in{i:06d}.jpg")
    IG = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

    idx_bg_prev_avg = (mask_prev_avg_cons == 0)
    BG_average_cons[idx_bg_prev_avg] = (
        alpha * IG[idx_bg_prev_avg] +
        (1 - alpha) * BG_average_cons[idx_bg_prev_avg]
    )

    diff_average_cons = cv2.absdiff(IG, BG_average_cons.astype(np.uint8))
    _, mask_average_cons = cv2.threshold(diff_average_cons, 10, 255, cv2.THRESH_BINARY)

    idx_bg_prev_med = (mask_prev_med_cons == 0)
    BG_median_cons[idx_bg_prev_med] = (
        BG_median_cons[idx_bg_prev_med]
        + (BG_median_cons[idx_bg_prev_med] < IG[idx_bg_prev_med]).astype(np.uint8)
        - (BG_median_cons[idx_bg_prev_med] > IG[idx_bg_prev_med]).astype(np.uint8)
    )

    diff_median_cons = cv2.absdiff(IG, BG_median_cons)
    _, mask_median_cons = cv2.threshold(diff_median_cons, 10, 255, cv2.THRESH_BINARY)


    if roi_start <= i <= roi_end:
        GT = cv2.imread(f"pedestrian/groundtruth/gt{i:06d}.png", cv2.IMREAD_GRAYSCALE)
        if GT is not None:
            tp, fp, fn = calculate_metrics(mask_average_cons, GT)
            TP_avg_cons += tp
            FP_avg_cons += fp
            FN_avg_cons += fn

            tp, fp, fn = calculate_metrics(mask_median_cons, GT)
            TP_med_cons += tp
            FP_med_cons += fp
            FN_med_cons += fn

    mask_prev_avg_cons = mask_average_cons
    mask_prev_med_cons = mask_median_cons

    show_all_in_one_row(
        IG,
        BG_average_cons,
        diff_average_cons,
        mask_average_cons,
        BG_median_cons,
        diff_median_cons,
        mask_median_cons
    )

summarize_metrics(TP_avg_cons, FP_avg_cons, FN_avg_cons, "AVERAGE (Conservative)")
summarize_metrics(TP_med_cons, FP_med_cons, FN_med_cons, "MEDIAN (Conservative)")


#3.4

In [ ]:
roi_path = 'pedestrian/temporalROI.txt'
with open(roi_path, 'r') as f:
    line = f.readline()
    roi_start, roi_end = line.split()
    roi_start = int(roi_start)
    roi_end = int(roi_end)

start_n = 400
N = 400

TP = FP = FN = 0

bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=16,
    detectShadows=False
)

for i in range(start_n, start_n+N, 1):
    current_frame_bgr = cv2.imread(f"pedestrian/input/in{i:06d}.jpg")
    IG = cv2.cvtColor(current_frame_bgr, cv2.COLOR_BGR2GRAY)

    fg_mask = bg_subtractor.apply(IG, learningRate=-1)

    mask = post_process(fg_mask)

    if roi_start <= i <= roi_end:
        GT = cv2.imread(f"pedestrian/groundtruth/gt{i:06d}.png", cv2.IMREAD_GRAYSCALE)
        if GT is not None:
            tp, fp, fn = calculate_metrics(mask, GT)
            TP += tp
            FP += fp
            FN += fn


summarize_metrics(TP, FP, FN, "GMM/MOG2")



GMM/MOG2 METRICS:
Precision: 0.6727
Recall:    0.8584
F1 Score:  0.7543


#3.5

In [ ]:
roi_path = 'pedestrian/temporalROI.txt'
with open(roi_path, 'r') as f:
    line = f.readline()
    roi_start_str, roi_end_str = line.strip().split()
    roi_start = int(roi_start_str)
    roi_end   = int(roi_end_str)

start_n = 400
N = 400

TP = FP = FN = 0

bg_subtractor = cv2.createBackgroundSubtractorKNN(
    history=500,
    dist2Threshold=400.0,
    detectShadows=False
)

for i in range(start_n, start_n+N, 1):
    current_frame_bgr = cv2.imread(f"pedestrian/input/in{i:06d}.jpg")
    IG = cv2.cvtColor(current_frame_bgr, cv2.COLOR_BGR2GRAY)

    fg_mask = bg_subtractor.apply(IG, learningRate=-1)

    mask = post_process(fg_mask)

    if roi_start <= i <= roi_end:
        GT = cv2.imread(f"pedestrian/groundtruth/gt{i:06d}.png", cv2.IMREAD_GRAYSCALE)
        if GT is not None:
            tp, fp, fn = calculate_metrics(mask, GT)
            TP += tp
            FP += fp
            FN += fn

summarize_metrics(TP, FP, FN, "KNN")


KNN METRICS:
Precision: 0.4801
Recall:    0.8923
F1 Score:  0.6243
